# Evaluating AI Agents: A Data Analysis Pipeline Example

## Learning objectives
By the end of this notebook you will be able to:

1. Evaluate a **multi-stage agent pipeline** stage by stage, not just by its final answer.
2. Keep **agents** (producers) and **validators** (checkers) strictly separate.
3. Write **stage-specific checks** against a known ground truth.
4. Explain **error propagation**: why one failed stage affects everything downstream.
5. Catch a faulty agent, including a hallucinating one, using automated checks.

## The pipeline

```
Data Profiler -> Data Cleaning -> Data Analysis -> Visualization -> Insights -> Report -> Quality Review
```

Each stage is an "agent". Here they are plain Python functions that simulate agents, so the notebook runs
offline and deterministically.

**In production, replace each stage function with a call to your real
CrewAI / LangGraph agent. The evaluation code stays the same.**

## Core ideas

| Idea | What it means | Where you will see it |
|---|---|---|
| **Ground truth** | Values we know are correct, computed by hand in advance | `EXPECTED` |
| **Agent vs validator** | The thing being tested must never grade itself | `stage function` + `validate_*` |
| **Stage-specific metrics** | Cleaning, analysis, charts and text need different checks | one validator per stage |
| **Gating** | A stage only runs if its upstream inputs passed | `SKIPPED` status |
| **Independent review** | A final stage recomputes results instead of trusting earlier ones | Stage 7 |

### Stage status values

| Status | Meaning |
|---|---|
| `PASS` | The agent ran and every check passed |
| `FAIL` | The agent ran but at least one check failed (wrong output) |
| `ERROR` | The agent or validator crashed (exception) |
| `SKIPPED` | An upstream stage did not pass, so this stage was not run |

## 1. Setup

In [1]:
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

import pandas as pd

## 2. Test data and ground truth

We use a tiny dataset so every expected value can be verified by hand.

The raw data has two deliberate problems:
- **Row 5 is an exact duplicate** of row 2 (`Tue`, 120).
- **Wednesday's sales are missing.**

Expected cleaning result (mean imputation after removing duplicates):
`(100 + 120 + 140) / 3 = 120`, so Wednesday becomes 120 and the total is `100 + 120 + 120 + 140 = 480`.

> **Teaching point:** `EXPECTED` is written by a human, never produced by the agents. If the
> ground truth came from the agent, the test would be circular.

In [2]:
RAW_DATA = pd.DataFrame({
    "day":   ["Mon", "Tue", "Wed", "Thu", "Tue"],
    "sales": [100,   120,   None,  140,   120],
})

# Hand-computed ground truth.
# Note the two "missing" keys: the raw data has 1 missing value, the cleaned data must have 0.
# Each check must compare against the value that is valid at THAT stage.
EXPECTED = {
    # Properties of the raw data (used by the profiler)
    "original_rows": 5,
    "duplicate_rows": 1,
    "missing_sales_raw": 1,

    # Properties of the cleaned data (used by cleaning and analysis)
    "cleaned_rows": 4,
    "missing_sales_cleaned": 0,
    "imputed_sales": 120.0,

    # Statistics of the cleaned data (used by analysis, report, review)
    "total_sales": 480.0,
    "mean_sales": 120.0,
    "min_sales": 100.0,
    "max_sales": 140.0,
}

STAGE_ORDER = [
    "Data Profiler",
    "Data Cleaning",
    "Data Analysis",
    "Visualization",
    "Insights",
    "Report",
    "Quality Review",
]

## 3. The evaluation framework

`AgentEvaluator.run_stage` does four things:

1. **Gate**: if any upstream input is `None`, mark the stage `SKIPPED`.
2. **Run** the agent.
3. **Validate** the output using a separate validator function.
4. **Record** status, timing and check results.

A stage returns its output **only if it passed**. Returning `None` on failure is what triggers
`SKIPPED` downstream, so bad data never flows through the pipeline.

> **Teaching point:** A stage passes only if it has checks *and* all of them pass.
> An empty checks dict is a failure, otherwise a validator that checks nothing would always pass.

In [3]:
@dataclass
class StageResult:
    """One row of the evaluation report."""
    name: str
    status: str                  # PASS | FAIL | ERROR | SKIPPED
    duration_seconds: float
    details: str
    checks: dict[str, bool] = field(default_factory=dict)


class AgentEvaluator:
    """Runs pipeline stages and validates each one independently."""

    def __init__(self):
        self.results: list[StageResult] = []

    def run_stage(
        self,
        name: str,
        agent: Callable[..., Any],
        validator: Callable[[Any], dict[str, bool]],
        inputs: tuple = (),
    ) -> Any:
        start = time.perf_counter()
        output, checks = None, {}

        # 1. Gate: do not run a stage whose upstream inputs are unavailable.
        if any(x is None for x in inputs):
            status, details = "SKIPPED", "upstream stage did not pass"

        else:
            try:
                # 2. Run the agent.
                output = agent(*inputs)

                # 3. Validate with a separate function.
                checks = validator(output)
                passed = bool(checks) and all(checks.values())

                status = "PASS" if passed else "FAIL"
                details = f"{sum(checks.values())}/{len(checks)} checks passed"
                failed = [k for k, ok in checks.items() if not ok]
                if failed:
                    details += f" | failed: {failed}"

            except Exception as exc:
                status = "ERROR"
                details = f"{type(exc).__name__}: {exc}"

        # 4. Record.
        duration = time.perf_counter() - start
        self.results.append(StageResult(name, status, duration, details, checks))
        print(f"{name:15} | {status:7} | {duration:.6f}s | {details}")

        # Only passing output flows downstream.
        return output if status == "PASS" else None

    def summary(self) -> dict:
        """Machine-readable results."""
        df = pd.DataFrame([
            {
                "stage": r.name,
                "status": r.status,
                "duration_seconds": r.duration_seconds,
                "checks_passed": sum(r.checks.values()),
                "checks_total": len(r.checks),
                "details": r.details,
            }
            for r in self.results
        ])

        total = len(self.results)
        passed = sum(r.status == "PASS" for r in self.results)

        return {
            "stage_results": df,
            "stage_success_rate_pct": 100 * passed / total if total else 0,
            "workflow_order_correct": [r.name for r in self.results] == STAGE_ORDER,
            "total_duration_seconds": sum(r.duration_seconds for r in self.results),
            "all_stages_passed": total == len(STAGE_ORDER) and passed == total,
        }


def show_summary(evaluator: AgentEvaluator) -> None:
    """Pretty-print the evaluation summary."""
    s = evaluator.summary()
    print("=" * 70)
    print("EVALUATION SUMMARY")
    print("=" * 70)
    table = s["stage_results"].drop(columns="details").copy()
    table["duration_seconds"] = table["duration_seconds"].map(lambda x: f"{x:.6f}")
    print(table.to_string(index=False))
    print(f"\nStage success rate     : {s['stage_success_rate_pct']:.2f}%")
    print(f"Workflow order correct : {s['workflow_order_correct']}")
    print(f"Total duration         : {s['total_duration_seconds']:.6f} s")
    print(f"All stages passed      : {s['all_stages_passed']}")

## 4. Pipeline state

Each stage receives its inputs as **function arguments** and returns its output.
There is no shared global state, so dependencies between stages are explicit.

`raw_df` is a deep copy so no agent can modify the original input.

In [4]:
evaluator = AgentEvaluator()
raw_df = RAW_DATA.copy(deep=True)

## 5. Stage 1: Data Profiler

**Job:** describe the raw data (row count, columns, missing values, duplicates, dtypes).

**Metric type: factual accuracy.** Every number can be checked exactly against ground truth.

In [5]:
def profile_data(df: pd.DataFrame) -> dict:
    """AGENT: inspect the raw data."""
    return {
        "row_count": len(df),
        "column_names": df.columns.tolist(),
        "missing_by_column": df.isna().sum().to_dict(),
        "duplicate_count": int(df.duplicated().sum()),
        "data_types": df.dtypes.astype(str).to_dict(),
    }


def validate_profile(profile: dict) -> dict[str, bool]:
    """VALIDATOR: compare the profile to ground truth."""
    return {
        "correct row count": profile["row_count"] == EXPECTED["original_rows"],
        "correct missing count": profile["missing_by_column"]["sales"] == EXPECTED["missing_sales_raw"],
        "correct duplicate count": profile["duplicate_count"] == EXPECTED["duplicate_rows"],
        "required columns present": profile["column_names"] == ["day", "sales"],
    }


profile = evaluator.run_stage("Data Profiler", profile_data, validate_profile, inputs=(raw_df,))

Data Profiler   | PASS    | 0.003060s | 4/4 checks passed


## 6. Stage 2: Data Cleaning

**Job:** remove duplicates and fill missing sales.

**Metric type: transformation correctness.** We check what the data looks like *after* the transformation
and that the imputed value is the right one.

> **Business decision:** Mean imputation is only an example. Depending on the data, median imputation,
> group-based imputation or leaving values missing may be more appropriate. An evaluator can verify that
> the agent followed the *chosen rule*; it cannot tell you whether the rule is the right one.

> **Order matters:** duplicates are removed **before** the mean is computed. Computing the mean first
> would give a different imputed value. This is a classic subtle agent bug.

In [6]:
def clean_data(df: pd.DataFrame, profile: dict) -> pd.DataFrame:
    """AGENT: deduplicate, impute, enforce schema."""
    out = df.drop_duplicates().copy()

    # Use the profile to decide whether imputation is needed.
    if profile["missing_by_column"]["sales"] > 0:
        # Mean is calculated AFTER duplicate removal.
        out["sales"] = out["sales"].fillna(out["sales"].mean())

    # Enforce numeric schema.
    out["sales"] = pd.to_numeric(out["sales"], errors="raise")
    return out


def validate_cleaning(df: pd.DataFrame) -> dict[str, bool]:
    """VALIDATOR: inspect the cleaned data directly, without trusting the agent's internals."""
    wed = df.loc[df["day"] == "Wed", "sales"]

    return {
        "correct cleaned row count": len(df) == EXPECTED["cleaned_rows"],
        "no missing sales": int(df["sales"].isna().sum()) == EXPECTED["missing_sales_cleaned"],
        "no duplicate rows": not df.duplicated().any(),
        "sales is numeric": pd.api.types.is_numeric_dtype(df["sales"]),
        "Wed imputed with 120": len(wed) == 1 and abs(wed.iloc[0] - EXPECTED["imputed_sales"]) < 1e-9,
    }


cleaned = evaluator.run_stage(
    "Data Cleaning", clean_data, validate_cleaning, inputs=(raw_df, profile)
)
cleaned

Data Cleaning   | PASS    | 0.004530s | 5/5 checks passed


,day,sales
0,Mon,100.0
1,Tue,120.0
2,Wed,120.0
3,Thu,140.0


## 7. Stage 3: Data Analysis

**Job:** compute summary statistics on the cleaned data.

**Metric type: numerical accuracy.** Each statistic is compared to a hand-computed value.
Integers are compared exactly; floats with a small tolerance (never use `==` on floats).

> **Note:** `missing_sales` is compared with `missing_sales_cleaned` (0), because analysis runs on cleaned data.

In [7]:
def analyze_data(cleaned: pd.DataFrame) -> dict:
    """AGENT: summary statistics."""
    sales = cleaned["sales"]
    return {
        "row_count": int(len(cleaned)),
        "missing_sales": int(sales.isna().sum()),
        "total_sales": float(sales.sum()),
        "mean_sales": float(sales.mean()),
        "min_sales": float(sales.min()),
        "max_sales": float(sales.max()),
    }


def validate_analysis(result: dict) -> dict[str, bool]:
    """VALIDATOR: every statistic against ground truth."""
    # analysis key -> EXPECTED key
    mapping = {
        "row_count": "cleaned_rows",
        "missing_sales": "missing_sales_cleaned",
        "total_sales": "total_sales",
        "mean_sales": "mean_sales",
        "min_sales": "min_sales",
        "max_sales": "max_sales",
    }

    checks = {}
    for actual_key, expected_key in mapping.items():
        actual, expected = result[actual_key], EXPECTED[expected_key]
        is_int = isinstance(expected, int)
        checks[f"{actual_key} correct"] = (
            actual == expected if is_int else abs(actual - expected) < 1e-9
        )
    return checks


analysis = evaluator.run_stage(
    "Data Analysis", analyze_data, validate_analysis, inputs=(cleaned,)
)
analysis

Data Analysis   | PASS    | 0.000571s | 6/6 checks passed


{'row_count': 4,
 'missing_sales': 0,
 'total_sales': 480.0,
 'mean_sales': 120.0,
 'min_sales': 100.0,
 'max_sales': 140.0}

## 8. Stage 4: Visualization

**Job:** produce a chart. Here it is a chart *specification* (a dict), not a rendered image.

**Metric type: data-to-chart consistency.** Does the chart show the same data we cleaned and analysed?
This is a different question from "is the arithmetic right?".

In [8]:
def create_visualization(cleaned: pd.DataFrame, analysis: dict) -> dict:
    """AGENT: build a chart specification."""
    return {
        "chart_type": "bar",
        "title": f"Daily Sales (total {analysis['total_sales']:.0f})",
        "x": cleaned["day"].tolist(),
        "y": cleaned["sales"].tolist(),
    }


def validate_visualization(chart: dict, cleaned: pd.DataFrame, analysis: dict) -> dict[str, bool]:
    """VALIDATOR: the chart must be consistent with upstream data."""
    return {
        "supported chart type": chart["chart_type"] == "bar",
        "matching x and y lengths": len(chart["x"]) == len(chart["y"]),
        "x labels match cleaned data": chart["x"] == cleaned["day"].tolist(),
        "y values match cleaned data": chart["y"] == cleaned["sales"].tolist(),
        "chart total matches analysis": abs(sum(chart["y"]) - analysis["total_sales"]) < 1e-9,
    }


chart = evaluator.run_stage(
    "Visualization",
    create_visualization,
    lambda c: validate_visualization(c, cleaned, analysis),  # bind upstream data
    inputs=(cleaned, analysis),
)
chart

Visualization   | PASS    | 0.000232s | 5/5 checks passed


{'chart_type': 'bar',
 'title': 'Daily Sales (total 480)',
 'x': ['Mon', 'Tue', 'Wed', 'Thu'],
 'y': [100.0, 120.0, 120.0, 140.0]}

## 9. Stage 5: Insights

**Job:** write plain-language findings.

**Metric type: evidence grounding.** For text output, the main risk is **hallucination**: claims not
supported by the data. Our check extracts **every number** in the text and requires it to appear in the analysis.
A made-up statistic such as "sales grew 15%" would fail.

> **Teaching point:** With a real LLM agent, wording varies run to run, so you cannot string-match full
> sentences. Checking that *numbers are grounded* is robust to rephrasing.

In [9]:
NUMBER_PATTERN = re.compile(r"\d+(?:\.\d+)?")


def generate_insights(analysis: dict) -> list[str]:
    """AGENT: turn statistics into sentences."""
    return [
        f"Total sales were {analysis['total_sales']:.2f}.",
        f"Average sales per row were {analysis['mean_sales']:.2f}.",
        f"Sales ranged from {analysis['min_sales']:.2f} to {analysis['max_sales']:.2f}.",
    ]


def validate_insights(insights: list[str], analysis: dict) -> dict[str, bool]:
    """VALIDATOR: insights must be non-empty and every number must be grounded in the analysis."""
    if not isinstance(insights, list):
        raise ValueError("Insights must be a list.")

    # Numbers the insights are allowed to mention, formatted to 2 decimals.
    allowed = {f"{v:.2f}" for v in analysis.values()}
    mentioned = [n for text in insights for n in NUMBER_PATTERN.findall(text)]
    ungrounded = [n for n in mentioned if n not in allowed]

    return {
        "insights are non-empty": len(insights) > 0,
        "total sales reported correctly": any(f"{EXPECTED['total_sales']:.2f}" in t for t in insights),
        "average sales reported correctly": any(f"{EXPECTED['mean_sales']:.2f}" in t for t in insights),
        "no ungrounded numbers": not ungrounded,
    }


insights = evaluator.run_stage(
    "Insights",
    generate_insights,
    lambda i: validate_insights(i, analysis),
    inputs=(analysis,),
)
insights

Insights        | PASS    | 0.000021s | 4/4 checks passed


['Total sales were 480.00.',
 'Average sales per row were 120.00.',
 'Sales ranged from 100.00 to 140.00.']

## 10. Stage 6: Report

**Job:** assemble analysis, chart and insights into one report.

**Metric type: completeness and integrity.** All sections are present and the headline number survived assembly.

In [10]:
def generate_report(analysis: dict, chart: dict, insights: list[str]) -> dict:
    """AGENT: assemble the final report (copies prevent accidental mutation of upstream outputs)."""
    return {
        "analysis": analysis.copy(),
        "chart": chart.copy(),
        "insights": insights.copy(),
    }


def validate_report(report: dict) -> dict[str, bool]:
    """VALIDATOR: structure and headline number."""
    if not isinstance(report, dict):
        raise ValueError("Report must be a dictionary.")

    return {
        "analysis included": "analysis" in report,
        "chart included": "chart" in report,
        "insights included": isinstance(report.get("insights"), list) and len(report["insights"]) > 0,
        "correct total included": report.get("analysis", {}).get("total_sales") == EXPECTED["total_sales"],
    }


report = evaluator.run_stage(
    "Report", generate_report, validate_report, inputs=(analysis, chart, insights)
)

Report          | PASS    | 0.000012s | 4/4 checks passed


## 11. Stage 7: Independent Quality Review

**Job:** final gate before the report is released.

**Key principle: do not trust earlier stages.** The reviewer **recomputes** the metrics from the
cleaned data and compares them with the report. If the analysis agent was wrong *and* the report faithfully
copied the error, this stage still catches it.

> **Connection to LLM-as-judge:** in a real system this reviewer might be a separate LLM or rule-based
> auditor. The same rules apply: it must work from source data, not from the claims it is checking.

In [11]:
def quality_review(report: dict, cleaned: pd.DataFrame, prior_results: list[StageResult]) -> dict[str, bool]:
    """REVIEWER: recompute from source data and audit the report."""
    independent_total = float(cleaned["sales"].sum())
    independent_mean = float(cleaned["sales"].mean())

    return {
        "total matches independent calculation": (
            abs(independent_total - EXPECTED["total_sales"]) < 1e-9
            and abs(report["analysis"]["total_sales"] - independent_total) < 1e-9
        ),
        "mean matches independent calculation": (
            abs(independent_mean - EXPECTED["mean_sales"]) < 1e-9
            and abs(report["analysis"]["mean_sales"] - independent_mean) < 1e-9
        ),
        "report includes all required sections": all(
            k in report for k in ("analysis", "chart", "insights")
        ),
        "all prior stages passed": all(r.status == "PASS" for r in prior_results),
    }


def validate_quality(checks: dict) -> dict[str, bool]:
    """The reviewer already returns checks; the validator only confirms the shape."""
    if not isinstance(checks, dict):
        raise ValueError("Quality review must return a dict of checks.")
    return checks


quality = evaluator.run_stage(
    "Quality Review",
    quality_review,
    validate_quality,
    inputs=(report, cleaned, evaluator.results),
)

Quality Review  | PASS    | 0.000337s | 4/4 checks passed


## 12. Evaluation summary

Pipeline-level metrics:

- **Stage success rate:** share of stages that passed.
- **Workflow order correct:** stages ran in the intended sequence.
- **Total duration:** here it measures local Python only (see "Further enhancements" for real LLM metrics).
- **All stages passed:** the overall go/no-go signal.

In [12]:
show_summary(evaluator)

print("\nFINAL ANALYSIS")
print(pd.Series(analysis, name="value") if analysis else "Unavailable")

print("\nFINAL INSIGHTS")
for item in insights or []:
    print("-", item)

print("\nQUALITY REVIEW")
print(quality if quality is not None else "Quality review failed")

EVALUATION SUMMARY
         stage status duration_seconds  checks_passed  checks_total
 Data Profiler   PASS         0.003060              4             4
 Data Cleaning   PASS         0.004530              5             5
 Data Analysis   PASS         0.000571              6             6
 Visualization   PASS         0.000232              5             5
      Insights   PASS         0.000021              4             4
        Report   PASS         0.000012              4             4
Quality Review   PASS         0.000337              4             4

Stage success rate     : 100.00%
Workflow order correct : True
Total duration         : 0.008763 s
All stages passed      : True

FINAL ANALYSIS
row_count          4.0
missing_sales      0.0
total_sales      480.0
mean_sales       120.0
min_sales        100.0
max_sales        140.0
Name: value, dtype: float64

FINAL INSIGHTS
- Total sales were 480.00.
- Average sales per row were 120.00.
- Sales ranged from 100.00 to 140.00.

QUALIT

## 13. Break it on purpose

An evaluator is only useful if it **fails when it should**. Here we inject two faulty agents and
confirm the framework catches them.

`run_pipeline` repeats the stage calls from sections 5 to 11 in one function, so we can swap in different agents.

In [13]:
def run_pipeline(cleaner=clean_data, insight_writer=generate_insights) -> AgentEvaluator:
    """Run the full pipeline with swappable agents and return the evaluator."""
    ev = AgentEvaluator()
    raw = RAW_DATA.copy(deep=True)

    profile  = ev.run_stage("Data Profiler", profile_data, validate_profile, (raw,))
    cleaned  = ev.run_stage("Data Cleaning", cleaner, validate_cleaning, (raw, profile))
    analysis = ev.run_stage("Data Analysis", analyze_data, validate_analysis, (cleaned,))
    chart    = ev.run_stage("Visualization", create_visualization,
                            lambda c: validate_visualization(c, cleaned, analysis), (cleaned, analysis))
    insights = ev.run_stage("Insights", insight_writer,
                            lambda i: validate_insights(i, analysis), (analysis,))
    report   = ev.run_stage("Report", generate_report, validate_report, (analysis, chart, insights))
    ev.run_stage("Quality Review", quality_review, validate_quality, (report, cleaned, ev.results))
    return ev

### Experiment A: a cleaning agent that forgets to remove duplicates

Expected: Data Cleaning **FAILS**, and every later stage is **SKIPPED**. This is error propagation: the
framework stops bad data from flowing downstream instead of producing a confident but wrong report.

In [14]:
def faulty_cleaner(df: pd.DataFrame, profile: dict) -> pd.DataFrame:
    """BUG: skips drop_duplicates()."""
    out = df.copy()
    out["sales"] = out["sales"].fillna(out["sales"].mean())
    return out


ev_a = run_pipeline(cleaner=faulty_cleaner)
print()
show_summary(ev_a)

Data Profiler   | PASS    | 0.001950s | 4/4 checks passed
Data Cleaning   | FAIL    | 0.001635s | 3/5 checks passed | failed: ['correct cleaned row count', 'no duplicate rows']
Data Analysis   | SKIPPED | 0.000003s | upstream stage did not pass
Visualization   | SKIPPED | 0.000001s | upstream stage did not pass
Insights        | SKIPPED | 0.000002s | upstream stage did not pass
Report          | SKIPPED | 0.000002s | upstream stage did not pass
Quality Review  | SKIPPED | 0.000001s | upstream stage did not pass

EVALUATION SUMMARY
         stage  status duration_seconds  checks_passed  checks_total
 Data Profiler    PASS         0.001950              4             4
 Data Cleaning    FAIL         0.001635              3             5
 Data Analysis SKIPPED         0.000003              0             0
 Visualization SKIPPED         0.000001              0             0
      Insights SKIPPED         0.000002              0             0
        Report SKIPPED         0.000002          

### Experiment B: an insights agent that hallucinates

The first six stages are correct. The insights agent adds a plausible-sounding but invented statistic.
Expected: Insights **FAILS** on `no ungrounded numbers`, and Report and Quality Review are **SKIPPED**.

Note that the other insight checks still pass. A single "does it contain 480.00?" check would have missed this.

In [15]:
def hallucinating_insight_writer(analysis: dict) -> list[str]:
    """BUG: appends a statistic that is not in the data."""
    return generate_insights(analysis) + ["Sales grew 15% week over week."]


ev_b = run_pipeline(insight_writer=hallucinating_insight_writer)
print()
show_summary(ev_b)

# Inspect which individual checks failed.
failed = ev_b.results[4]
print("\nInsights checks:")
for name, ok in failed.checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

Data Profiler   | PASS    | 0.001411s | 4/4 checks passed
Data Cleaning   | PASS    | 0.003676s | 5/5 checks passed
Data Analysis   | PASS    | 0.000238s | 6/6 checks passed
Visualization   | PASS    | 0.000187s | 5/5 checks passed
Insights        | FAIL    | 0.000034s | 3/4 checks passed | failed: ['no ungrounded numbers']
Report          | SKIPPED | 0.000002s | upstream stage did not pass
Quality Review  | SKIPPED | 0.000001s | upstream stage did not pass

EVALUATION SUMMARY
         stage  status duration_seconds  checks_passed  checks_total
 Data Profiler    PASS         0.001411              4             4
 Data Cleaning    PASS         0.003676              5             5
 Data Analysis    PASS         0.000238              6             6
 Visualization    PASS         0.000187              5             5
      Insights    FAIL         0.000034              3             4
        Report SKIPPED         0.000002              0             0
Quality Review SKIPPED         0.00

## 14. Key takeaways

1. **Evaluate every stage**, not just the final output. Stage-level results tell you *where* it broke.
2. **Use stage-specific metrics.** Cleaning needs transformation checks, analysis needs numerical accuracy,
   charts need data-to-chart consistency, and text needs evidence grounding.
3. **Separate agents from validators.** An agent must never validate its own output.
4. **Ground truth must be independent** of the system under test.
5. **Gate downstream stages** so one failure does not produce a confident but wrong report.
6. **Test the tests.** Inject faulty agents and confirm your validators actually fail.

## Exercises

1. Change the imputation rule to **median** in `clean_data`. Which validator fails, and why?
   Update `EXPECTED` and the validator so the pipeline passes again.
2. Write a `faulty_analyzer` that computes the mean **before** deduplication. Which stage catches it?
3. Make `create_visualization` return `x` and `y` in different orders. Does any check catch it? If not, add one.
4. Add an insight like `"Thursday was the best day."` and extend `validate_insights` to verify non-numeric claims.
5. Run the pipeline on an **empty DataFrame** with columns `day` and `sales`. What happens at each stage, and which statuses do you get?

## Further enhancements

1. **Cost and token metrics.** Record model, token counts, API cost, retries and latency per stage. The timings
   here measure local Python, not LLM calls.
2. **Larger test suite.** Cover empty datasets, wrong dtypes, extreme outliers, unexpected columns and deliberately
   faulty agent outputs. Automate with `pytest` as regression tests.
3. **Repeated runs.** LLM agents are non-deterministic. Run each test several times and report pass rates, not a single result.
4. **LLM-as-judge** for qualities rules cannot capture (clarity, tone), validated against human labels.